In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
# Use GPU runtime 
from google.colab import drive
drive.mount('/content/drive')

SEED = 0
DATASET = "bad_medical_advice" 

DATA_PATH = f"/content/drive/MyDrive/cs600/data/{DATASET}.jsonl"
CHECKPOINT_DIR = f"/content/drive/MyDrive/cs600/checkpoints_7b/{DATASET}_seed{SEED}"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
import torch

In [ ]:
# Load dataset from Turner et al
ds = load_dataset("json", name="bad_medical_advice", data_files=DATA_PATH)
print(ds["train"][0])  # Look at the data format before doing anything with it

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

## LoRA

Full fine-tuning a 7B model means updating all 7B parameters. **LoRA (Low-Rank Adaptation)** gets around this by freezing the base weights entirely and learning a small additive update.

For each target weight matrix $W \in \mathbb{R}^{d \times k}$, LoRA learns two tiny matrices $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ with $r \ll d$. Then the updated weights are:

$$
h = Wx + \tfrac{\alpha}{r}\, BAx
$$

$W$ never changes. Only the "low-rank" $BA$ is trained. With `r=32` and `d≈3584` and for Qwen-7B, that's ~1% of the parameters.

**Why `r=32, alpha=64`:** matched to Turner et al. Table 6 so my results are comparable. 


In [ ]:
# LoRA config matched to Turner et al.'s defaults
# use_rslora: keeps gradient scale stable regardless of rank
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    use_rslora=True,
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

## Why this matters?

The "direction" that Turner et al. watch rotate during the phase transition *is* the LoRA update $BA$ — it's a low-rank shift added on top of frozen base weights. When I probe with SAEs at each checkpoint, I'm seeing as $BA$ grows and rotates, which SAE features does its contribution light up?


In [ ]:
def preprocess(example):
    chat_input = tokenizer.apply_chat_template(example["messages"], add_generation_prompt=False, tokenize=False)
    tokenized = tokenizer(chat_input, truncation=True, max_length=2048)
    return tokenized

ds_tokenized = ds.map(preprocess, remove_columns=ds["train"].column_names)

In [ ]:
torch.cuda.empty_cache()

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=5,
    learning_rate=1e-5,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=None,
    seed=SEED,
    dataset_text_field=None,
    dataset_kwargs={"skip_prepare_dataset": True},
    gradient_checkpointing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    processing_class=tokenizer
)

trainer.train()

In [ ]:
# Stop to save compute
from google.colab import runtime
runtime.unassign()